# Gjeng
Vi kan definere en **gjeng** som et sett av noder som "henger sammen" men som henger løst sammen med resten.  

## Girvan-Newman

Vi kan bruke [Girvan-Newman](https://en.wikipedia.org/wiki/Girvan%E2%80%93Newman_algorithm) til å finne slike gjenger.  

Konseptet med en bronode er at mange korteste-sti passerer gjennom noden (som gjør den sentral sett i lys av informasjonsspredning).  Vi kan flytte fokus fra nodene til kantene, og sette vekt på kantene basert på hvor mange korteste-sti som går gjennom dem.  Ved å fjerne kanter med høy vekt fragmenteres grafen, og gjengene står igjen.  De som først separeres ut er de som har bronoder som har få (men viktige) forbindelser til resten

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

In [ ]:
with open("lesmiserables-character-network/parsed_data/jean-complete-node.csv", "r") as fd:
    noder_rå = fd.readlines()
#
# Bort med første linje
noder_rå = noder_rå[1:]

# Formatet er 
# "TH","Thénardier","Thénardier, innkeeper in Montfermeil, aka Jondrette"\n
# Hent ut det vi trenger, og lag noder

G = nx.Graph()
for n in noder_rå:
    _s = n.split('"')
    G.add_node(_s[1], Navn=_s[3], Rolle=_s[5])
#
print(f"Antall noder i grafen: {G.number_of_nodes()}")

# Kantene
with open("lesmiserables-character-network/parsed_data/jean-complete-edge.csv", "r") as fd:
    kanter_rå = fd.readlines()
#
# Bort med første
kanter_rå = kanter_rå[1:]
print(f"Antall kanter: {len(kanter_rå)}")
#Formatet er 
# "MY","NP","Undirected","1","1.1.1" 
# hvor 1.1.1 er kapittelet hvor forbindelsen opptrer
for k in kanter_rå:
    _s = k.split(",")
    _ = G.add_edge(_s[0][1:3], _s[1][1:3])

In [ ]:
from networkx.algorithms.community import girvan_newman

# Denne deler grafen i stadig flere gjenger (itererer)
generator = girvan_newman(G)
første = next(generator)
andre = next(generator)
tredje = next(generator)
fjerde = next(generator)

# Håper det er færre gjenger enn det er farger
fargeliste = ["blue", "brown", "orange", "pink", "green", "gray", "red", "olive", "purple", "cyan"]

sett_id4 = {}
for snummer, s in enumerate(fjerde):
    for e in s:
        sett_id4[e] = snummer
    #
#
node_farge4 = []
for node in G:
    idx = sett_id4.get(node)
    node_farge4.append(fargeliste[idx])
#

sett_id3 = {}
for snummer, s in enumerate(tredje):
    for e in s:
        sett_id3[e] = snummer
    #
#
node_farge3 = []
for node in G:
    idx = sett_id3.get(node)
    node_farge3.append(fargeliste[idx])
#

# Tegne dem ved siden av hverandre
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(15, 10))

pos = nx.spring_layout(G, seed=10) # Beste seed så langt
nx.draw(G, pos, ax=axes[0], with_labels=False, node_color=node_farge4, font_size=12, node_size=75)
nx.draw(G, pos, ax=axes[1], with_labels=False, node_color=node_farge3, font_size=12, node_size=75)
plt.show()

## Louvain

Vi kan se på en gjeng som noder som er tettere knyttet til hverandre enn til resten (de andre nodene).  Det vil si at om vi tar utgangspunkt i en tilfeldig graf (se nedenfor) så vil det være flere kanten i en gjeng enn vi forventer; tettheten er større.

Noder flyttes inn i grupper, og man ser om tettheten "omkring" går opp eller ned når noden flyttes fra en gjeng til en annen.  Man sitter igjen med den optimale delingen i gjenger.

Ulempen er at gjengene er gjensidig utelukkende; neppe representativ for virkeligheten.  Videre, når to nærliggende klikker slås sammen kan det godt tenkes at det er "godt" for den overordnede inndelingen, men at de to i utgangspunktet var tett knyttet går tapt.

kjøretiden er kun $O(n \log n)$ og kan derfor brukes også på svært store grafer.
Info [på Wikipedia](https://en.wikipedia.org/wiki/Louvain_method).

In [ ]:
# Community detection with the Louvain Method
from networkx.algorithms.community import louvain_communities

# Calculate best partition for Louvain method
partitions = louvain_communities(G)
print(f"Fant {len(partition)} gjenger")

# Finn de best forbindne i hver gjeng
ledere = []
for p in partitions:
    leder = max(p, key=lambda n:G.degree(n))
    ledere += [leder]
#
print(f"Lederne er: {ledere}")

# Håper det er færre gjenger enn det er farger
fargeliste = ["blue", "brown", "orange", "pink", "green", "gray", "red", "olive", "purple", "cyan"]
node_farge= []
størrelse = 75
node_størrelse = []
# Forutsetter at rekkefølgen er den samme hver gang
for node in G:
    # Hvilken partisjon er denne noden i?
    for idx, l in enumerate(partitions):
        if node in l:
            node_farge.append(fargeliste[idx])
            break
        #
    # Større noder for ledere
    if node in ledere:
        node_størrelse.append(størrelse* 10)
    else:
        node_størrelse.append(størrelse)
    #
#


# Vis frem
plt.figure(figsize=(10, 10))
pos = nx.spring_layout(G, seed=10)
nx.draw(G, pos, with_labels=False, node_color=node_farge, font_size=12, node_size=node_størrelse)
plt.title("Les Mmisérables med Louvian gjenger")
plt.show()